# 第1章 Notebook：1次元フィンの放熱

対応章: [`../chapters/01_stegosaurus_heat_basic.md`](../chapters/01_stegosaurus_heat_basic.md)

この notebook は、卒業研究準備セミナーの数値実験用である。上から順に実行すれば、本文で説明した図を再現できる。設定パラメータは上部のセルにまとめてある。乱数は seed を固定している。

## 1. ライブラリ読み込み

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

## 2. パラメータ設定（ここを変えて実験する）

In [ ]:
k     = 20.0    # thermal conductivity [W/(m K)]
h     = 25.0    # convection coefficient [W/(m^2 K)]
T_b   = 40.0    # base temperature [degC]
T_air = 25.0    # air temperature [degC]
width = 0.30    # plate width [m]
thick = 0.03    # plate thickness [m]
L     = 0.40    # fin length [m]

A_c = width * thick           # cross-section area
P   = 2 * (width + thick)     # cross-section perimeter
m   = np.sqrt(h * P / (k * A_c))
theta_b = T_b - T_air
print(f'm = {m:.3f} [1/m], mL = {m*L:.3f}')

## 3. 解析解と差分法による温度分布

解析解 $\theta(x)=\theta_b\cosh(m(L-x))/\cosh(mL)$ と、差分法で解いた線形方程式 $A\theta=b$ を比較する。

In [ ]:
def theta_analytic(x, L, m, theta_b):
    return theta_b * np.cosh(m * (L - x)) / np.cosh(m * L)

def theta_fd(L, m, theta_b, n=101):
    x = np.linspace(0, L, n)
    dx = x[1] - x[0]
    A = np.zeros((n, n))
    b = np.zeros(n)
    A[0, 0] = 1.0; b[0] = theta_b              # base: theta(0)=theta_b
    for i in range(1, n - 1):                   # interior nodes
        A[i, i - 1] = 1.0
        A[i, i] = -2.0 - (m * dx) ** 2
        A[i, i + 1] = 1.0
    A[-1, -1] = 1.0; A[-1, -2] = -1.0; b[-1] = 0.0  # insulated tip
    return x, np.linalg.solve(A, b)

x = np.linspace(0, L, 101)
theta_exact = theta_analytic(x, L, m, theta_b)
x_fd, theta_num = theta_fd(L, m, theta_b)
print('max |analytic - FD| =', np.max(np.abs(theta_num - theta_analytic(x_fd, L, m, theta_b))))

## 4. 温度分布の図

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(x, theta_exact, label='analytic')
plt.plot(x_fd, theta_num, 'o', ms=3, label='finite difference')
plt.xlabel('position x [m]'); plt.ylabel('excess temp theta [K]')
plt.title('1D fin temperature distribution')
plt.legend(); plt.tight_layout(); plt.show()

## 5. フィン長を変えた比較：放熱量と効率

放熱量 $Q=\sqrt{hPkA_c}\,\theta_b\tanh(mL)$、効率 $\eta=\tanh(mL)/mL$。

In [ ]:
def Q_fin(L, m, theta_b, k, P, A_c):
    return np.sqrt(h * P * k * A_c) * theta_b * np.tanh(m * L)

Ls = np.linspace(0.05, 1.0, 50)
mL = m * Ls
Q = Q_fin(Ls, m, theta_b, k, P, A_c)
eta = np.tanh(mL) / mL

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(mL, Q); ax[0].set_xlabel('mL'); ax[0].set_ylabel('Q_fin [W]')
ax[0].set_title('heat dissipation vs mL')
ax[1].plot(mL, eta); ax[1].set_xlabel('mL'); ax[1].set_ylabel('efficiency eta')
ax[1].set_title('fin efficiency vs mL')
plt.tight_layout(); plt.show()

## 6. 放熱量の近似計算（代表値）

In [ ]:
print(f'current design: L={L} m, mL={m*L:.2f}, Q={Q_fin(L,m,theta_b,k,P,A_c):.2f} W, eta={np.tanh(m*L)/(m*L):.3f}')

## 7. 課題（自分で変更する）

1. `mL` が 0.1, 1, 3, 10 になるよう `L` を選び、`Q_fin` と `eta` を比較せよ。
2. `h`（対流係数）を 2 倍にすると温度分布と放熱量はどう変わるか、上のセルを変更して確かめよ。

In [ ]:
# === 課題セル ===
for target in [0.1, 1, 3, 10]:
    Ltmp = target / m
    print(f'mL={target:>4}: L={Ltmp:.3f} m, Q={Q_fin(Ltmp,m,theta_b,k,P,A_c):.2f} W, eta={np.tanh(target)/target:.3f}')